In [ ]:
# Boosted CNN for 1D Light Curves
# (No extra data; aug + better arch + TTA)
# ================================
import os, pickle, math
from pathlib import Path
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, precision_recall_curve, roc_curve,
    classification_report
)
from sklearn.utils.class_weight import compute_class_weight

# -------------------------------
# Config
# -------------------------------
DATA_PATH = "tess_koi_lc_rescued_clip8.npz"  
OUT_DIR   = Path("models")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
BATCH_SIZE = 64
EPOCHS = 120
BASE_LR = 5e-4
TARGET_PREC = 0.95   # for high-precision threshold
LABEL_SMOOTH = 0.05  # label smoothing

# for augmentation
MAX_SHIFT = 6        # how many LC bins to shift left/right (phase jitter)
NOISE_STD = 0.01     # light Gaussian noise
AMP_JITTER = 0.02    # amplitude scaling ±2%

# -------------------------------
# Utils
# -------------------------------
def set_all_seeds(seed=42):
    np.random.seed(seed)
    tf.random.set_seed(seed)

def summarize(y_true, prob, thr=0.5):
    pred = (prob >= thr).astype(int)
    return {
        "Accuracy": float(accuracy_score(y_true, pred)),
        "Precision": float(precision_score(y_true, pred, zero_division=0)),
        "Recall": float(recall_score(y_true, pred, zero_division=0)),
        "F1": float(f1_score(y_true, pred, zero_division=0)),
        "ROC_AUC": float(roc_auc_score(y_true, prob)),
        "CM": confusion_matrix(y_true, pred).tolist()
    }

def tune_thr_for_precision(y_true, prob, target=0.95):
    p, r, t = precision_recall_curve(y_true, prob)
    best = {"Thr": 0.5, "Precision": 0.0, "Recall": 0.0}
    for P, R, T in zip(p[:-1], r[:-1], t):
        if P >= target and R > best["Recall"]:
            best = {"Thr": float(T), "Precision": float(P), "Recall": float(R)}
    return best

def best_thr_youden(y_true, prob):
    fpr, tpr, thr = roc_curve(y_true, prob)
    j = tpr - fpr
    idx = int(np.argmax(j))
    return float(thr[idx])

# -------------------------------
# Data load + split (group by ids)
# -------------------------------
set_all_seeds(SEED)

data = np.load(DATA_PATH, allow_pickle=True)
X   = data["X"]        # shape: (N, L, 1) or (N, L)
y   = data["y"].astype("int32")
ids = data["ids"] if "ids" in data else np.arange(len(X))

# ensure shape [N, L, 1]
if X.ndim == 2:
    X = X[:, :, None]
elif X.ndim == 3 and X.shape[-1] != 1:
    if X.shape[1] == 1:  # (N,1,L) -> (N,L,1)
        X = np.transpose(X, (0,2,1))

print("Loaded:", X.shape, "class dist:", np.bincount(y))

gss = GroupShuffleSplit(n_splits=100, test_size=0.2, random_state=SEED)
ok = False
for tr_idx, va_idx in gss.split(X, y, groups=ids):
    if len(np.unique(y[tr_idx])) == 2 and len(np.unique(y[va_idx])) == 2:
        ok = True
        break
assert ok, "No binary split found."

X_tr, X_va = X[tr_idx], X[va_idx]
y_tr, y_va = y[tr_idx], y[va_idx]
ids_tr, ids_va = ids[tr_idx], ids[va_idx]

print(f"Train: {X_tr.shape}, dist:", np.bincount(y_tr))
print(f"Valid: {X_va.shape}, dist:", np.bincount(y_va))

# class weights (for minor imbalance)
w = compute_class_weight(class_weight='balanced', classes=np.array([0,1]), y=y_tr)
class_weight = {0: float(w[0]), 1: float(w[1])}
print("class_weight:", class_weight)

# -------------------------------
# tf.data pipelines with augmentation
# -------------------------------
L = X_tr.shape[1]

def aug_fn(x, y):
    # x: [L,1]
    # 1) phase jitter (roll)
    shift = tf.random.uniform([], minval=-MAX_SHIFT, maxval=MAX_SHIFT+1, dtype=tf.int32)
    x = tf.roll(x, shift=shift, axis=0)

    # 2) amplitude jitter
    alpha = 1.0 + tf.random.uniform([], -AMP_JITTER, AMP_JITTER)
    x = x * tf.cast(alpha, x.dtype)

    # 3) gaussian noise
    noise = tf.random.normal(tf.shape(x), stddev=NOISE_STD, dtype=x.dtype)
    x = x + noise
    return x, y

def ds_from_numpy(Xn, yn, training=True, batch=BATCH_SIZE):
    ds = tf.data.Dataset.from_tensor_slices((Xn, yn))
    if training:
        ds = ds.shuffle(len(Xn), seed=SEED, reshuffle_each_iteration=True)
        ds = ds.map(aug_fn, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = ds_from_numpy(X_tr, y_tr, training=True)
val_ds   = ds_from_numpy(X_va, y_va, training=False)

# -------------------------------
# Model (stronger CNN + BN + SpatialDropout)
# -------------------------------
def build_cnn_final(input_len=201, lr=BASE_LR, use_noise=True, label_smoothing=LABEL_SMOOTH):
    inp = keras.Input(shape=(input_len, 1))
    x = inp

    if use_noise:
        x = layers.GaussianNoise(0.01)(x)

    # Block 1
    x = layers.Conv1D(32, 11, padding="same", activation=None,
                      kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPool1D(2)(x)            # 201 -> ~100
    x = layers.SpatialDropout1D(0.05)(x)

    # Block 2
    x = layers.Conv1D(64, 7, padding="same", activation=None)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPool1D(2)(x)            # ~100 -> ~50
    x = layers.SpatialDropout1D(0.10)(x)

    # Block 3 (deeper)
    x = layers.Conv1D(128, 5, padding="same", activation=None)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPool1D(2)(x)            # ~50 -> ~25
    x = layers.SpatialDropout1D(0.15)(x)

    # Head
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, activation="relu")(x)
    out = layers.Dense(1, activation="sigmoid")(x)

    model = keras.Model(inp, out)
    model.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss=keras.losses.BinaryCrossentropy(label_smoothing=label_smoothing),
        metrics=[keras.metrics.AUC(name="auc"),
                 keras.metrics.Precision(name="prec"),
                 keras.metrics.Recall(name="rec")]
    )
    return model

model = build_cnn_final(input_len=X_tr.shape[1])

callbacks = [
    keras.callbacks.ModelCheckpoint(
        OUT_DIR / "best_cnn.keras", save_best_only=True,
        monitor="val_auc", mode="max"
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_auc", mode="max", patience=15,
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_auc", mode="max", factor=0.5,
        patience=7, min_lr=1e-6
    ),
]

# -------------------------------
# Train
# -------------------------------
print("\nTraining ...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weight,
    verbose=2,
)
print("Done.")

# -------------------------------
# Evaluation + TTA
# -------------------------------
def predict_tta(model, X, shifts=(-6,-4,-2,0,2,4,6), batch=256):
    # average predictions across phase shifts
    Xs = []
    for s in shifts:
        Xs.append(np.roll(X, shift=s, axis=1))
    X_stack = np.concatenate(Xs, axis=0)         # [k*N, L, 1]
    p_stack = model.predict(X_stack, batch_size=batch, verbose=0).ravel()
    k = len(shifts)
    p_mean = p_stack.reshape(k, len(X)).mean(axis=0)
    return p_mean

# base (without TTA)
p_va_base = model.predict(X_va, batch_size=256).ravel()
m_base = summarize(y_va, p_va_base, thr=0.5)
t_hp    = tune_thr_for_precision(y_va, p_va_base, target=TARGET_PREC)
thr_y   = best_thr_youden(y_va, p_va_base)
m_hp    = summarize(y_va, p_va_base, thr=t_hp["Thr"])
m_y     = summarize(y_va, p_va_base, thr=thr_y)

# with TTA
p_va_tta = predict_tta(model, X_va)
m_base_tta = summarize(y_va, p_va_tta, thr=0.5)
t_hp_tta   = tune_thr_for_precision(y_va, p_va_tta, target=TARGET_PREC)
thr_y_tta  = best_thr_youden(y_va, p_va_tta)
m_hp_tta   = summarize(y_va, p_va_tta, thr=t_hp_tta["Thr"])
m_y_tta    = summarize(y_va, p_va_tta, thr=thr_y_tta)

print("\n== VAL @0.5 ==",
      {k: round(v,4) for k,v in m_base.items() if k!='CM'})
print("== VAL tuned (Precision>={}) ==".format(TARGET_PREC),
      {k: round(v,4) for k,v in t_hp.items()}, "|",
      {k: round(v,4) for k,v in m_hp.items() if k!='CM'})
print("== VAL tuned (Youden) ==", {"Thr": round(thr_y,4)}, "|",
      {k: round(v,4) for k,v in m_y.items() if k!='CM'})

print("\n== VAL+TTA @0.5 ==",
      {k: round(v,4) for k,v in m_base_tta.items() if k!='CM'})
print("== VAL+TTA tuned (Precision>={}) ==".format(TARGET_PREC),
      {k: round(v,4) for k,v in t_hp_tta.items()}, "|",
      {k: round(v,4) for k,v in m_hp_tta.items() if k!='CM'})
print("== VAL+TTA tuned (Youden) ==", {"Thr": round(thr_y_tta,4)}, "|",
      {k: round(v,4) for k,v in m_y_tta.items() if k!='CM'})

print("\nClassification report @0.5 (TTA):\n",
      classification_report(y_va, (p_va_tta>=0.5).astype(int), digits=3))

# -------------------------------
# Save model + artifacts
# -------------------------------
final_model_path = OUT_DIR / "cnn_lc_boosted.keras"
model.save(final_model_path)

artifacts = {
    "model_path": str(final_model_path),
    "best_ckpt": str(OUT_DIR / "best_cnn.keras"),
    "input_len": int(X_tr.shape[1]),
    "val_ids": [str(x) for x in ids_va],
    "thr_default": 0.5,
    "thr_high_precision": float(t_hp_tta["Thr"]),
    "thr_youden": float(thr_y_tta),
    "val_metrics_base": m_base_tta,
    "val_metrics_tuned_precision": m_hp_tta,
    "val_metrics_youden": m_y_tta,
}
with open(OUT_DIR / "cnn_artifacts_boost.pkl", "wb") as f:
    pickle.dump(artifacts, f)

print("\nSaved model and artifacts to:", OUT_DIR.resolve())


Loaded: (600, 201, 1) class dist: [294 306]
Train: (480, 201, 1), dist: [240 240]
Valid: (120, 201, 1), dist: [54 66]
class_weight: {0: 1.0, 1: 1.0}

Training ...
Epoch 1/120
8/8 - 6s - 718ms/step - auc: 0.5609 - loss: 0.7077 - prec: 0.5628 - rec: 0.5042 - val_auc: 0.6048 - val_loss: 0.6889 - val_prec: 0.5377 - val_rec: 0.8636
Epoch 2/120
8/8 - 0s - 30ms/step - auc: 0.5464 - loss: 0.7197 - prec: 0.5359 - rec: 0.5292 - val_auc: 0.5243 - val_loss: 0.6974 - val_prec: 0.8000 - val_rec: 0.1818
Epoch 3/120
8/8 - 0s - 44ms/step - auc: 0.5668 - loss: 0.7106 - prec: 0.5371 - rec: 0.5125 - val_auc: 0.6911 - val_loss: 0.6866 - val_prec: 0.8889 - val_rec: 0.3636
Epoch 4/120
8/8 - 0s - 27ms/step - auc: 0.5749 - loss: 0.6938 - prec: 0.5539 - rec: 0.4708 - val_auc: 0.6909 - val_loss: 0.6891 - val_prec: 0.8667 - val_rec: 0.1970
Epoch 5/120
8/8 - 0s - 28ms/step - auc: 0.5797 - loss: 0.7080 - prec: 0.5700 - rec: 0.4917 - val_auc: 0.6923 - val_loss: 0.6889 - val_prec: 0.8824 - val_rec: 0.2273
Epoch 6/120